In [1]:
import sys
import os

# Move one directory up (to project root)
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("Added to PYTHONPATH:", project_root)

Added to PYTHONPATH: /vols/cms/mm1221/geant4sim/scripts/validation_new


In [2]:
import numpy as np
import pandas as pd
import awkward as ak

from src.data_loader.Data_Loader import Data_Loader
from src.models.model_loader import model_loader
from src.clustering.clusterer import clusterer
from src.metrics.build_dataframe import build_dataframe

In [3]:
loader = Data_Loader(
    root="/vols/cms/mm1221/geant4sim/simulations/build/Datasets/Test_EM_all/Test_pi.1001_1002.root",
    split="test",
    max_events=100,
    batch_size=1,
    shuffle=False,
    follow_batch=["x"],
)

### Loading ROOT file: /vols/cms/mm1221/geant4sim/simulations/build/Datasets/Test_EM_all/Test_pi.1001_1002.root
### Keeping only events with len(PrimaryEnergies) in (1, 5, 10, 15, 20, 25, 30)
Loaded 7 events (post-filter)


/cvmfs/sft.cern.ch/lcg/views/LCG_105a_cuda/x86_64-el9-gcc11-opt/lib/python3.9/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_config = {
    "task": "oc",
    "hidden_dim": 64,
    "num_layers": 3,
    "dropout": 0.01,
    "k": 24,

    "contrastive_dim": 16,
    "coord_dim": 4,
    "path" : "/vols/cms/mm1221/geant4sim/scripts/training/ObjectCondensation/runs/EM_2_10_CD4_delta5_1/best_model.pt"
}

model = model_loader(
    config=model_config,
    device = device
)

print(f"Model loaded successfully")


Model loaded successfully


In [5]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph, radius_neighbors_graph

import pandas as pd
from tqdm import tqdm


In [11]:
import torch.nn.functional as F
all_predictions = []
true_labels = []
all_energies = []

for i, data in enumerate(loader):
    data = data.to(device)

    out = model(data.x, data.x_batch)
    preds = out[1]                     
    #preds = F.normalize(preds, p=2, dim=1)
    all_predictions.append(preds)
    all_energies.append(data.x[:,3])
    true_labels.append(data.assoc)

    print(i)
    if i > 10:
        break

0
1
2
3
4
5
6


In [12]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# -----------------------------
# Per-event embedding metrics on a physics-relevant "energy core"
# - Build per-shower core: keep hits that account for keep_frac (e.g. 90%) of shower energy
# - Compute metrics on core hits only
# - Metrics:
#     Recall@1, Recall@10
#     Pairwise AUC via sampled pairs (same vs different group)
#     Intra hard: Q_intra of same-group cosine distances
#     Inter hard: Q_inter of different-group cosine distances
#     Optional: kNN contamination@10 (C@10)
# -----------------------------


def _energy_core_mask_per_event(
    y: torch.Tensor,
    E: torch.Tensor,
    *,
    keep_frac: float = 0.90,
    min_hits_per_group: int = 2,
) -> torch.Tensor:
    """
    Returns boolean mask (N,) selecting per-shower energy core hits:
      For each label, sort hits by descending energy and keep the smallest set
      whose cumulative energy reaches keep_frac of that group's total energy.

    If a group has < min_hits_per_group hits, it is ignored for intra pairs anyway.
    If total energy is 0, we keep all hits in that group (fallback).
    """
    y = y.detach().cpu().long().view(-1)
    E = E.detach().cpu().float().view(-1).clamp_min(0.0)

    N = y.numel()
    if E.numel() != N:
        raise ValueError(f"Energy length {E.numel()} != N {N}")

    mask = torch.zeros(N, dtype=torch.bool)

    for lab in torch.unique(y):
        idx = (y == lab).nonzero(as_tuple=False).view(-1)
        if idx.numel() == 0:
            continue

        Ei = E[idx]
        tot = float(Ei.sum().item())

        if tot <= 0.0:
            # no meaningful energy info -> keep all hits of this group
            mask[idx] = True
            continue

        order = torch.argsort(Ei, descending=True)
        idx_sorted = idx[order]
        Ei_sorted = Ei[order]
        csum_frac = torch.cumsum(Ei_sorted, dim=0) / tot

        keep = csum_frac <= keep_frac

        # ensure we keep at least min_hits_per_group if possible
        if keep.sum().item() < min_hits_per_group and idx_sorted.numel() >= min_hits_per_group:
            keep[:min_hits_per_group] = True
        elif keep.sum().item() == 0 and idx_sorted.numel() > 0:
            keep[0] = True

        mask[idx_sorted[keep]] = True

    return mask


def _sample_pairs_from_labels(
    y: torch.Tensor,
    n_pos: int = 2000,
    n_neg: int = 2000,
    rng=None,
):
    """
    Sample positive (same-label) and negative (different-label) index pairs.
    Positives: pick label uniformly among labels with >=2 hits (group-balanced),
               then sample two hits inside that label.
    Negatives: pick two labels uniformly, then one hit from each.
    Returns: (i_pos, j_pos, i_neg, j_neg) as 1D Long tensors.
    """
    if rng is None:
        rng = np.random.default_rng(123)

    y_np = y.detach().cpu().numpy()
    labels = np.unique(y_np)

    by_label = {lab: np.where(y_np == lab)[0] for lab in labels}
    pos_labels = [lab for lab in labels if by_label[lab].size >= 2]

    # positives
    if len(pos_labels) == 0 or n_pos <= 0:
        i_pos = j_pos = np.empty((0,), dtype=np.int64)
    else:
        i_pos = np.empty((n_pos,), dtype=np.int64)
        j_pos = np.empty((n_pos,), dtype=np.int64)
        for t in range(n_pos):
            lab = rng.choice(pos_labels)
            idx = by_label[lab]
            a, b = rng.choice(idx, size=2, replace=False)
            i_pos[t], j_pos[t] = a, b

    # negatives
    if labels.size < 2 or n_neg <= 0:
        i_neg = j_neg = np.empty((0,), dtype=np.int64)
    else:
        i_neg = np.empty((n_neg,), dtype=np.int64)
        j_neg = np.empty((n_neg,), dtype=np.int64)
        for t in range(n_neg):
            la, lb = rng.choice(labels, size=2, replace=False)
            a = rng.choice(by_label[la])
            b = rng.choice(by_label[lb])
            i_neg[t], j_neg[t] = a, b

    return (
        torch.from_numpy(i_pos).long(),
        torch.from_numpy(j_pos).long(),
        torch.from_numpy(i_neg).long(),
        torch.from_numpy(j_neg).long(),
    )


def _auc_from_scores(pos_scores: np.ndarray, neg_scores: np.ndarray) -> float:
    """AUC = P(score_pos > score_neg) + 0.5 P(equal)."""
    if pos_scores.size == 0 or neg_scores.size == 0:
        return np.nan
    diff = pos_scores[:, None] - neg_scores[None, :]
    return float((diff > 0).mean() + 0.5 * (diff == 0).mean())


@torch.no_grad()
def _recall_at_k_cosine(z: torch.Tensor, y: torch.Tensor, k: int) -> float:
    """Recall@k within event using full cosine similarity matrix."""
    N = z.size(0)
    if N <= 1:
        return np.nan

    sim = z @ z.t()
    sim.fill_diagonal_(-1e9)

    kk = min(k, N - 1)
    nn_idx = torch.topk(sim, k=kk, dim=1, largest=True).indices
    hit = (y[nn_idx] == y[:, None]).any(dim=1)
    return float(hit.float().mean().item())


@torch.no_grad()
def _knn_contamination(z: torch.Tensor, y: torch.Tensor, K: int = 10) -> float:
    """
    C@K (kNN contamination):
      for each node, fraction of its top-K neighbors that are from a different label;
      then group-balance by averaging per label (so big showers don't dominate).
    """
    N = z.size(0)
    if N <= 1:
        return np.nan

    K_eff = min(K, N - 1)
    if K_eff <= 0:
        return np.nan

    sim = z @ z.t()
    sim.fill_diagonal_(-1e9)

    nn_idx = torch.topk(sim, k=K_eff, dim=1, largest=True).indices
    same = (y[nn_idx] == y[:, None]).float()
    contam_per_node = 1.0 - same.mean(dim=1)

    vals = []
    for lab in torch.unique(y):
        mask = (y == lab)
        if mask.any():
            vals.append(contam_per_node[mask].mean())
    if len(vals) == 0:
        return np.nan
    return float(torch.stack(vals).mean().item())


def _quantile(x: np.ndarray, q: float) -> float:
    if x.size == 0:
        return np.nan
    return float(np.quantile(x, q))


@torch.no_grad()
def compute_embedding_metrics_dataframe(
    all_predictions,
    true_labels,
    all_energies,
    *,
    n_pos_pairs: int = 2000,
    n_neg_pairs: int = 2000,
    seed: int = 123,
    # Tail-risk knobs
    intra_q: float = 0.90,     # high quantile of same-group distances
    inter_q: float = 0.10,     # low quantile of diff-group distances
    include_c10: bool = True,
    # Energy-core selection
    keep_energy_frac: float = 0.90,
):
    """
    Compute per-event metrics on the "energy core" hits only.

    all_predictions: list of (N,D) embeddings per event
    true_labels:     list of (N,) truth group ids per event
    all_energies:    list of (N,) hit energies per event (same ordering as embeddings/labels)

    Returns DataFrame with per-event rows.
    """
    if not (len(all_predictions) == len(true_labels) == len(all_energies)):
        raise ValueError("all_predictions, true_labels, all_energies must have same length")

    rng = np.random.default_rng(seed)
    rows = []

    intra_name = f"intra_q{int(round(100*intra_q))}_core{int(round(100*keep_energy_frac))}"
    inter_name = f"inter_q{int(round(100*inter_q))}_core{int(round(100*keep_energy_frac))}"

    for event_id, (emb, y, E) in enumerate(zip(all_predictions, true_labels, all_energies)):
        if not torch.is_tensor(emb):
            emb = torch.tensor(emb)
        if not torch.is_tensor(y):
            y = torch.tensor(y)
        if not torch.is_tensor(E):
            E = torch.tensor(E)

        emb = emb.detach().cpu().float()
        y = y.detach().cpu().long().view(-1)
        E = E.detach().cpu().float().view(-1)

        N_all = emb.size(0)
        if y.numel() != N_all or E.numel() != N_all:
            raise ValueError(
                f"Event {event_id}: emb N={N_all}, labels N={y.numel()}, energies N={E.numel()}"
            )

        n_groups_all = int(torch.unique(y).numel())

        # --- build energy-core mask and filter ---
        core_mask = _energy_core_mask_per_event(y, E, keep_frac=keep_energy_frac)

        emb_c = emb[core_mask]
        y_c = y[core_mask]

        N = emb_c.size(0)
        if N <= 1 or torch.unique(y_c).numel() < 1:
            # not enough points to compute meaningful metrics
            rows.append(
                dict(
                    event_id=int(event_id),
                    n_nodes_all=int(N_all),
                    n_groups_all=int(n_groups_all),
                    n_nodes=int(N),
                    n_groups=int(torch.unique(y_c).numel()) if N > 0 else 0,
                    r_at_1=np.nan,
                    r_at_10=np.nan,
                    auc=np.nan,
                    **{intra_name: np.nan, inter_name: np.nan},
                    c_at_10=np.nan if include_c10 else None,
                )
            )
            continue

        z = F.normalize(emb_c, p=2, dim=1)
        n_groups = int(torch.unique(y_c).numel())

        # Recall@K on core hits
        r1 = _recall_at_k_cosine(z, y_c, k=1)
        r10 = _recall_at_k_cosine(z, y_c, k=10)

        # Sample pairs on core hits
        i_pos, j_pos, i_neg, j_neg = _sample_pairs_from_labels(
            y_c, n_pos=n_pos_pairs, n_neg=n_neg_pairs, rng=rng
        )

        if i_pos.numel() > 0:
            s_pos = (z[i_pos] * z[j_pos]).sum(dim=1).numpy()
            d_pos = (1.0 - s_pos)
        else:
            s_pos = np.array([], dtype=np.float32)
            d_pos = np.array([], dtype=np.float32)

        if i_neg.numel() > 0:
            s_neg = (z[i_neg] * z[j_neg]).sum(dim=1).numpy()
            d_neg = (1.0 - s_neg)
        else:
            s_neg = np.array([], dtype=np.float32)
            d_neg = np.array([], dtype=np.float32)

        auc = _auc_from_scores(s_pos, s_neg)

        intra_hard = _quantile(d_pos, intra_q)  # higher = worse
        inter_hard = _quantile(d_neg, inter_q)  # lower = worse (confusable)

        row = dict(
            event_id=int(event_id),
            n_nodes_all=int(N_all),
            n_groups_all=int(n_groups_all),
            n_nodes=int(N),
            n_groups=int(n_groups),
            r_at_1=float(r1),
            r_at_10=float(r10),
            auc=float(auc),
            **{intra_name: float(intra_hard), inter_name: float(inter_hard)},
        )

        if include_c10:
            row["c_at_10"] = float(_knn_contamination(z, y_c, K=10))

        rows.append(row)

    df = pd.DataFrame(rows)
    # if include_c10=False we might have put None in some rows; clean that
    if not include_c10 and "c_at_10" in df.columns:
        df = df.drop(columns=["c_at_10"])
    return df


# -------- usage ----------
df_metrics = compute_embedding_metrics_dataframe(
    all_predictions,
    true_labels,
    all_energies,
    n_pos_pairs=2000,
    n_neg_pairs=2000,
    intra_q=0.5,
    inter_q=0.5,
    include_c10=True,
    keep_energy_frac=1.0,
)
# print(df_metrics.head())

In [13]:
df_metrics

,event_id,n_nodes_all,n_groups_all,n_nodes,n_groups,r_at_1,r_at_10,auc,intra_q50_core100,inter_q50_core100,c_at_10
0,0,11508,10,11506,10,0.794977,0.953329,0.939024,0.012810,0.971831,0.230473
1,1,6654,5,6652,5,0.797504,0.960012,0.885531,0.033746,1.248596,0.236354
2,2,23245,25,23240,25,0.599613,0.897246,0.915934,0.022358,0.796143,0.431736
3,3,1656,1,1655,1,1.000000,1.000000,NaN,0.129543,NaN,0.000000
4,4,411,1,411,1,1.000000,1.000000,NaN,0.017823,NaN,0.000000
5,5,14623,15,14618,15,0.612464,0.915720,0.925986,0.023053,1.014630,0.423309
6,6,3742,5,3742,5,0.830037,0.960449,0.941044,0.007551,1.204188,0.166536


In [11]:
df_metrics

,event_id,n_nodes_all,n_groups_all,n_nodes,n_groups,r_at_1,r_at_10,auc,intra_q99_core80,inter_q1_core80,c_at_10
0,0,22767,16,5510,16,0.787477,0.925408,0.925703,0.789505,0.008312,0.226958
1,1,41920,27,10164,27,0.644727,0.890889,0.922195,0.965426,0.012185,0.378233
2,2,32300,22,7884,22,0.684297,0.903095,0.924173,0.868741,0.010932,0.360742


In [43]:
df_metrics

,event_id,n_nodes_all,n_groups_all,n_nodes,n_groups,r_at_1,r_at_10,auc,intra_q99_core95,inter_q1_core95,c_at_10
0,0,16014,16,6893,16,0.774264,0.942260,0.953856,0.643918,0.005205,0.255048
1,1,25180,27,10808,27,0.698834,0.920059,0.952971,0.814336,0.001761,0.316454
2,2,20683,22,8927,22,0.733169,0.930996,0.957612,0.639927,0.001645,0.307953


In [16]:
df_metrics

,event_id,n_nodes_all,n_groups_all,n_nodes,n_groups,r_at_1,r_at_10,auc,intra_q99_core90,inter_q1_core90,c_at_10
0,0,22767,16,9740,16,0.751848,0.918275,0.910635,0.812065,0.008008,0.266269
1,1,41920,27,17868,27,0.584565,0.874804,0.912564,0.873106,0.007158,0.436185
2,2,32300,22,13888,22,0.640913,0.892497,0.917768,0.831976,0.013823,0.415909
